# About this Notebook
- This is the main interface for running Nanocraft.
- Each section is one main functionality of the toolkit. Please fill and adapt them to fit your needs.
- A description of the keywords for each function is available in the documentation.

# Load the environment
- Please execute the following cell each time you start the Notebook.
- Each time you modify one of the configuration files in `./misc/`, you have to restart the Notebook kernel.

In [ ]:
%load_ext autoreload 
%autoreload 2
import sys
# sys.path.append('/path/to/nanocraft/top/directory') # path to directory containing the src folder; uncomment if notebook used outside of Nanocraft directory
from src.nanocraft.cnfg import *
from src.nanocraft.log_cnfg import *

# Supercell & nanocrystal cutting
- Load a supercell prepared externally (`.xyz`) or load a `.cif` file.
- If you provide a `.cif` file, you must choose the size of the supercell.

Unless you choose a specific central site, the supercell is centered on its COM. 
A consequence is that, if N is even and M is odd, then N$\times$N$\times$N and M$\times$M$\times$M *are not* the same.

**Please provide `rcut` in nm.**

In [ ]:
from src.nanocraft.sprcll import load_supercell, apply_supercell_transform

inputfile = ""
supercell_size = [,,]

supercell = load_supercell(inputfile=inputfile, supercell_size=supercell_size)

In [ ]:
from src.nanocraft.ncgen import generate 
from src.nanocraft.utls import grid, run_parallel
from tqdm import tqdm

output_dir = None

settings = {
    'rcut': .0, # nm
    'fratio': .0, 
    'mask_shape': "",
    'central_site': "",
    'acceptance_criterion': ""
}

settings_grid = grid(settings)

for settings in tqdm(settings_grid, desc="Progress"):
    generate(settings=settings, supercell=supercell, output_dir=output_dir)

# run_parallel(generate, settings_grid, n_workers=8, supercell=supercell, output_dir=output_dir) # parallel execution

# Passivation
- Satisfy dangling bonds of the system at its surface.
- All dangling bonds are passivated, after which some passivating groups are removed so that the overall oxidation state of the system remains neutral.

In [ ]:
from src.nanocraft.pssv import passivate

output_dir = None
nrep = 1

settings = {
    'nanocrystal_file': "",
    'passiv_scheme': "",
    'binding_site':  "",
    'facet_constrain': "",
    'coord_constrain': "",
    'neutral_reach': "",
    'oxidation_method': "",
}

settings_grid = grid(settings, duplicate=nrep)

for settings in tqdm(settings_grid, desc="Progress"):
    passivate(settings=settings, output_dir=output_dir)

# Shuffle bonds / atoms
- Random displacement (normal distribution) of atoms / bonds.
    - Atoms are translated in a sphere around their initial position.
    - Bonds are stretched positively or negatively along their internuclear axis.

In [ ]:
from src.nanocraft.shffl import shuffle

nrep = 1
output_dir = None

settings = {
    "xyz_file": "",
    "layer": "",
    "atoms_bonds_set": "",
    "percent_shuffle": .0,
    "max_disp": .0,
    "random_seed": True,
}

settings_grid = grid(settings, duplicate=nrep)

for settings in tqdm(settings_grid, desc="Progress"):
    shuffle(settings=settings, output_dir=output_dir)

# Atomic susbtitution / vacancy
- Substitute one of the core atom by a new one to study vacancies, alloys, shell composition, etc.
- To create vacancies, leave the replacement atom an empty string.

In [ ]:
from src.nanocraft.srflyrs import substitute

nrep = 1
output_dir = None

settings = {
    'nanocrystal_file': "",
    'atom_to_replace': "",
    'replacement_atom': "",
    'mixing_ratio': .0,
    'method': "",
    'direction': "",
    'random_seed': True
}

settings_grid = grid(settings, duplicate=nrep)

for settings in tqdm(settings_grid, desc="Progress"):
    substitute(settings=settings, output_dir=output_dir)

# Functionalization
- Replace a molecular fragment (passivation group or ligand) by a new ligand.
- The new ligands must be prepared so that its principal molecular axis is aligned with the z-axis in the .xyz file.

**If you wish to place ligands on a bare structure, use the `functionalize_bare` method and not `functionalize`**.

In [ ]:
from src.nanocraft.molutls import align_zaxis_xyplane, align_zaxis
from src.nanocraft.ioxyz import load_xyz, save_xyz

inputfile  = "/path/to/ligand/ligand.xyz" # ligand with one dangling bond at the end of the chain ('zat1')
outputfile = inputfile.split('.xyz')[0] + '_aligned.xyz'

ligand = load_xyz(inputfile)
ligand = align_zaxis_xyplane(mol=ligand, zat1='S', zat2='H', xat1='C')

save_xyz(ligand, outputfile)

In [ ]:
from src.nanocraft.fnctn import functionalize

nrep = 1
output_dir = None

settings = {
    'ligand_file': "",
    'nanocrystal_file': "",
    'density_type': "",
    'density_value': .0,
    'anchor_fragment': "",
    'placement_method': "",
    'random_angle': True,
    'random_seed': True
}

settings_grid = grid(settings, duplicate=nrep)

for settings in tqdm(settings_grid, desc="Progress"):
    functionalize(settings=settings, output_dir=output_dir)

In [ ]:
from src.nanocraft.fnctn import functionalize_bare

nrep = 1
output_dir = None

settings = {
    'ligand_file': "",
    'nanocrystal_file': "",
    'density_type': "",
    'density_value': .0,
    'anchor_site': "",
    'anchor_type': 1,
    'placement_method': "",
    'nloc_COM': 11,
    'random_angle': True,
    'random_seed': True,
}

settings_grid = grid(settings, duplicate=nrep)

for settings in tqdm(settings_grid, desc="Progress"):
    functionalize_bare(settings=settings, output_dir=output_dir)